In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
import csv
import html
import random
import yaml
import pandas as pd2

import requests

# CSV, JSON, YAML

**Fred Brosman**

*05.05.2026*

# Ülesanne 1 — Test Trivia API abil
Looge programm, mis võimaldab kasutajal läbida testi, kasutades Trivia API küsimusi, ning salvestada tulemused JSON-faili.

Programm peab:

1. küsima kasutajalt kasutajanime
2. laadima Trivia API-st kategooriad
3. laskma kasutajal valida kategooria
4. küsima küsimuste arvu vahemikus 1–20
5. laskma valida küsimuste tüübi: `multiple` või `boolean`
6. tegema API päringu vastavalt kasutaja valikutele
7. kuvama küsimused ja vastusevariandid
8. kontrollima kasutaja vastuseid
9. kuvama testi lõpus skoori ja õiged vastused
10. salvestama tulemuse CSV-faili
11. kuvama tulemuste statistika: TOP 5 ja keskmine tulemus

Lahenduses kasutan:

- klassi `TriviaQuestion`, mis hoiab ühe küsimuse andmeid
- klassi `TriviaResult`, mis hoiab ühe testi tulemuse andmeid
- klassi `TriviaGame`, mis juhib testi läbimist
- eraldi funktsioone API päringute, CSV salvestamise ja statistika jaoks

## Konstandid

In [140]:
MIN_QUESTION_COUNT = 1
MAX_QUESTION_COUNT = 20

QUESTION_TYPES = {"multiple", "boolean"}

CATEGORIES_URL = "https://opentdb.com/api_category.php"
QUESTIONS_URL = "https://opentdb.com/api.php"

RESULTS_FILE = Path("trivia_tulemused.csv")

DEFAULT_TOP_COUNT = 5
REQUEST_TIMEOUT = 10

CSV_FIELDNAMES = [
    "kasutajanimi",
    "kategooria",
    "küsimuste_arv",
    "tüüp",
    "skoor",
    "kuupäev",
]

## 1.1 Klass TriviaQuestion

See klass kirjeldab ühte Trivia API-st saadud küsimust.

Igal küsimusel on:

- küsimuse tekst
- õige vastus
- valed vastused
- küsimuse tüüp
- kategooria

In [141]:
@dataclass
class TriviaQuestion:
    """
    Klass ühe Trivia küsimuse andmete hoidmiseks.

    Attributes
    ----------
    category : str
        Küsimuse kategooria.
    question_type : str
        Küsimuse tüüp: multiple või boolean.
    question_text : str
        Küsimuse tekst.
    correct_answer : str
        Õige vastus.
    incorrect_answers : list[str]
        Valede vastuste nimekiri.
    """

    category: str
    question_type: str
    question_text: str
    correct_answer: str
    incorrect_answers: list[str] = field(default_factory=list)

    def __post_init__(self) -> None:
        """
        Kontrollib, et küsimuse põhiandmed oleksid korrektsed.
        """
        if not self.category or not self.category.strip():
            raise ValueError("Kategooria ei tohi olla tühi.")

        if self.question_type not in QUESTION_TYPES:
            raise ValueError("Küsimuse tüüp peab olema 'multiple' või 'boolean'.")

        if not self.question_text or not self.question_text.strip():
            raise ValueError("Küsimuse tekst ei tohi olla tühi.")

        if not self.correct_answer or not self.correct_answer.strip():
            raise ValueError("Õige vastus ei tohi olla tühi.")

        self.category = self.category.strip()
        self.question_text = self.question_text.strip()
        self.correct_answer = self.correct_answer.strip()

    def get_shuffled_answers(self) -> list[str]:
        """
        Tagastab kõik vastusevariandid juhuslikus järjekorras.

        Returns
        -------
        list[str]
            Segatud vastusevariantide nimekiri.
        """
        answers = self.incorrect_answers + [self.correct_answer]
        random.shuffle(answers)
        return answers

    def is_correct(self, answer: str) -> bool:
        """
        Kontrollib, kas kasutaja vastus on õige.

        Parameters
        ----------
        answer : str
            Kasutaja valitud vastus.

        Returns
        -------
        bool
            True, kui vastus on õige. False, kui vastus on vale.
        """
        return answer == self.correct_answer

    def __str__(self) -> str:
        """
        Tagastab küsimuse loetava tekstiesituse.
        """
        return f"{self.category} | {self.question_text}"

## 1.2 Klass TriviaResult

See klass kirjeldab ühe testi tulemust.

Tulemusse salvestan:

- kasutajanime
- kategooria
- küsimuste arvu
- küsimuste tüübi
- skoori
- kuupäeva ja kellaaja

Seda klassi on hiljem mugav kasutada CSV-faili salvestamisel.

In [142]:
@dataclass
class TriviaResult:
    """
    Klass ühe testi tulemuse hoidmiseks.

    Attributes
    ----------
    username : str
        Mängija nimi.
    category : str
        Küsimuste kategooria.
    question_count : int
        Küsimuste arv.
    question_type : str
        Küsimuste tüüp.
    score : int
        Õigete vastuste arv.
    created_at : datetime
        Testi tegemise kuupäev ja kellaaeg.
    """

    username: str
    category: str
    question_count: int
    question_type: str
    score: int
    created_at: datetime = field(default_factory=datetime.now)

    def __post_init__(self) -> None:
        """
        Kontrollib, et tulemuse andmed oleksid korrektsed.
        """
        if not self.username or not self.username.strip():
            raise ValueError("Kasutajanimi ei tohi olla tühi.")

        if not self.category or not self.category.strip():
            raise ValueError("Kategooria ei tohi olla tühi.")

        if self.question_count < MIN_QUESTION_COUNT or self.question_count > MAX_QUESTION_COUNT:
            raise ValueError(
                f"Küsimuste arv peab olema vahemikus "
                f"{MIN_QUESTION_COUNT} kuni {MAX_QUESTION_COUNT}."
            )

        if self.question_type not in QUESTION_TYPES:
            raise ValueError("Küsimuse tüüp peab olema 'multiple' või 'boolean'.")

        if self.score < 0 or self.score > self.question_count:
            raise ValueError("Skoor ei saa olla väiksem kui 0 ega suurem kui küsimuste arv.")

        self.username = self.username.strip()
        self.category = self.category.strip()

    def percentage(self) -> float:
        """
        Arvutab tulemuse protsendina.

        Returns
        -------
        float
            Tulemus protsendina.
        """
        return round(self.score / self.question_count * 100, 2)

    def to_csv_row(self) -> dict[str, str | int]:
        """
        Teisendab tulemuse CSV-faili rea kujule.

        Returns
        -------
        dict
            Sõnastik, kus võtmed vastavad CSV veergudele.
        """
        return {
            "kasutajanimi": self.username,
            "kategooria": self.category,
            "küsimuste_arv": self.question_count,
            "tüüp": self.question_type,
            "skoor": self.score,
            "kuupäev": self.created_at.strftime("%Y-%m-%d %H:%M:%S")
        }

    def __str__(self) -> str:
        """
        Tagastab tulemuse loetava tekstiesituse.
        """
        return (
            f"{self.username} | "
            f"{self.category} | "
            f"{self.score}/{self.question_count} | "
            f"{self.percentage()}%"
        )

## 1.3 Klass TriviaGame

See klass juhib testi läbimist.

Klassis hoian:

- kasutajanime
- küsimuste nimekirja
- kasutaja vastuseid
- skoori

Klassi meetodid kuvavad küsimused, küsivad kasutaja vastust, kontrollivad vastust ja näitavad testi lõpus tulemuse.

In [143]:
@dataclass
class TriviaGame:
    """
    Klass Trivia testi läbiviimiseks.

    Attributes
    ----------
    username : str
        Mängija nimi.
    questions : list[TriviaQuestion]
        Trivia küsimuste nimekiri.
    user_answers : list[tuple[TriviaQuestion, str]]
        Kasutaja vastused koos küsimustega.
    score : int
        Õigete vastuste arv.
    """

    username: str
    questions: list[TriviaQuestion]
    user_answers: list[tuple[TriviaQuestion, str]] = field(default_factory=list)
    score: int = 0

    def __post_init__(self) -> None:
        """
        Kontrollib, et mängu põhiandmed oleksid korrektsed.
        """
        if not self.username or not self.username.strip():
            raise ValueError("Kasutajanimi ei tohi olla tühi.")

        if not self.questions:
            raise ValueError("Küsimuste nimekiri ei tohi olla tühi.")

        self.username = self.username.strip()

    def ask_one_question(self, question: TriviaQuestion, question_number: int) -> None:
        """
        Kuvab ühe küsimuse, küsib vastuse ja kontrollib seda.

        Parameters
        ----------
        question : TriviaQuestion
            Küsimus, mida kasutajale kuvatakse.
        question_number : int
            Küsimuse järjekorranumber.
        """
        answers = question.get_shuffled_answers()

        print()
        print(f"Küsimus {question_number}: {question.question_text}")

        for index, answer in enumerate(answers, start=1):
            print(f"{index}. {answer}")

        user_choice = self.ask_answer_number(len(answers))
        user_answer = answers[user_choice - 1]

        self.user_answers.append((question, user_answer))

        if question.is_correct(user_answer):
            self.score += 1
            print("Õige!")
        else:
            print(f"Vale. Õige vastus oli: {question.correct_answer}")

    def ask_answer_number(self, answer_count: int) -> int:
        """
        Küsib kasutajalt vastuse numbri ja kontrollib sisendit.

        Parameters
        ----------
        answer_count : int
            Mitu vastusevarianti küsimusel on.

        Returns
        -------
        int
            Kasutaja valitud vastuse number.
        """
        while True:
            try:
                choice = int(input("Vali vastuse number: "))

                if 1 <= choice <= answer_count:
                    return choice

                print("Sellist vastusevarianti ei ole.")

            except ValueError:
                print("Palun sisesta vastuse number.")

    def run(self) -> None:
        """
        Käivitab testi ja käib kõik küsimused läbi.
        """
        print(f"Alustame testi. Mängija: {self.username}")

        for index, question in enumerate(self.questions, start=1):
            self.ask_one_question(question, index)

    def print_score(self) -> None:
        """
        Kuvab testi lõppskoori.
        """
        print()
        print(f"Teie skoor: {self.score}/{len(self.questions)}")

    def print_correct_answers(self) -> None:
        """
        Kuvab testi lõpus kõik õiged vastused.
        """
        print()
        print("Õiged vastused:")

        for index, question in enumerate(self.questions, start=1):
            print(f"{index}. {question.question_text}")
            print(f"   Õige vastus: {question.correct_answer}")

    def create_result(self) -> TriviaResult:
        """
        Loob mängu põhjal TriviaResult objekti.

        Returns
        -------
        TriviaResult
            Testi tulemus.
        """
        first_question = self.questions[0]

        return TriviaResult(
            username=self.username,
            category=first_question.category,
            question_count=len(self.questions),
            question_type=first_question.question_type,
            score=self.score
        )

## 1.4 API funktsioonid

Selles osas loon funktsioonid, mis suhtlevad Open Trivia DB API-ga.

Vaja on kahte põhilist asja:

1. laadida kõik kategooriad
2. laadida küsimused kasutaja valikute põhjal

API tagastab andmed JSON-kujul. Pythonis saan neid kasutada sõnastike ja listidena.

In [144]:
def load_categories() -> list[dict]:
    """
    Laeb Open Trivia DB API-st kõik kategooriad.

    Returns
    -------
    list[dict]
        Kategooriate nimekiri. Igal kategoorial on id ja name.
    """
    try:
        response = requests.get(CATEGORIES_URL, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()

        return data["trivia_categories"]

    except requests.RequestException as error:
        raise RuntimeError("Kategooriate laadimine ebaõnnestus.") from error

    except ValueError as error:
        raise RuntimeError("API ei tagastanud korrektset JSON-vastust.") from error

    except KeyError as error:
        raise RuntimeError("API vastuses puudub kategooriate nimekiri.") from error


def print_categories(categories: list[dict]) -> None:
    """
    Väljastab kategooriad ekraanile.

    Parameters
    ----------
    categories : list[dict]
        Kategooriate nimekiri.
    """
    print("Saadaolevad kategooriad:")
    print()

    for category in categories:
        print(f'{category["id"]}: {category["name"]}')

## 1.5 Küsimuste laadimine API-st

Järgmine funktsioon teeb päringu küsimuste jaoks.

Kasutaja valib:

- kategooria ID
- küsimuste arvu
- küsimuse tüübi

API vastuses on küsimused koos õigete ja valede vastustega.  
Need teisendan `TriviaQuestion` objektideks.

In [145]:
def create_question_from_api_data(question_data: dict) -> TriviaQuestion:
    """
    Teisendab API-st saadud ühe küsimuse TriviaQuestion objektiks.

    Parameters
    ----------
    question_data : dict
        API-st saadud ühe küsimuse andmed.

    Returns
    -------
    TriviaQuestion
        Küsimuse objekt.
    """
    category = html.unescape(question_data["category"])
    question_type = html.unescape(question_data["type"])
    question_text = html.unescape(question_data["question"])
    correct_answer = html.unescape(question_data["correct_answer"])

    incorrect_answers = [
        html.unescape(answer)
        for answer in question_data["incorrect_answers"]
    ]

    return TriviaQuestion(
        category=category,
        question_type=question_type,
        question_text=question_text,
        correct_answer=correct_answer,
        incorrect_answers=incorrect_answers
    )


def fetch_questions(
    amount: int,
    category_id: int,
    question_type: str
) -> list[TriviaQuestion]:
    """
    Laeb API-st küsimused ja tagastab need TriviaQuestion objektidena.

    Parameters
    ----------
    amount : int
        Küsimuste arv.
    category_id : int
        Valitud kategooria ID.
    question_type : str
        Küsimuste tüüp: multiple või boolean.

    Returns
    -------
    list[TriviaQuestion]
        TriviaQuestion objektide nimekiri.
    """
    params = {
        "amount": amount,
        "category": category_id,
        "type": question_type
    }

    try:
        response = requests.get(
            QUESTIONS_URL,
            params=params,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        data = response.json()

    except requests.RequestException as error:
        raise RuntimeError("Küsimuste laadimine ebaõnnestus.") from error

    if data.get("response_code") != 0:
        raise ValueError(
            "API ei tagastanud küsimusi. "
            "Proovi teist kategooriat või väiksemat küsimuste arvu."
        )

    return [
        create_question_from_api_data(question_data)
        for question_data in data.get("results", [])
    ]

## 1.6 Kasutaja valikute küsimine

Selles osas loon funktsioonid, mis küsivad kasutajalt testi seaded.

Küsitakse:

- kasutajanimi
- kategooria ID
- küsimuste arv
- küsimuste tüüp

Iga funktsioon kontrollib ka sisendit, et programm vale sisendi korral kohe katki ei läheks.

In [146]:
def ask_username() -> str:
    """
    Küsib kasutajalt kasutajanime.

    Returns
    -------
    str
        Kasutaja sisestatud nimi.
    """
    while True:
        username = input("Sisesta kasutajanimi: ").strip()

        if username:
            return username

        print("Kasutajanimi ei tohi olla tühi.")


def ask_category(categories: list[dict]) -> tuple[int, str]:
    """
    Küsib kasutajalt kategooria ID ja kontrollib, kas see on olemas.

    Parameters
    ----------
    categories : list[dict]
        API-st saadud kategooriate nimekiri.

    Returns
    -------
    tuple[int, str]
        Valitud kategooria ID ja kategooria nimi.
    """
    valid_categories = {}

    for category in categories:
        valid_categories[category["id"]] = category["name"]

    while True:
        try:
            category_id = int(input("Sisesta kategooria ID: "))

            if category_id in valid_categories:
                return category_id, valid_categories[category_id]

            print("Sellist kategooriat ei ole nimekirjas.")

        except ValueError:
            print("Palun sisesta kategooria ID numbrina.")


def ask_question_count() -> int:
    """
    Küsib kasutajalt küsimuste arvu.

    Returns
    -------
    int
        Küsimuste arv vahemikus 1 kuni 20.
    """
    while True:
        try:
            question_count = int(
                input(f"Mitu küsimust soovid? ({MIN_QUESTION_COUNT}-{MAX_QUESTION_COUNT}): ")
            )

            if MIN_QUESTION_COUNT <= question_count <= MAX_QUESTION_COUNT:
                return question_count

            print(
                f"Küsimuste arv peab olema vahemikus "
                f"{MIN_QUESTION_COUNT} kuni {MAX_QUESTION_COUNT}."
            )

        except ValueError:
            print("Palun sisesta küsimuste arv numbrina.")


def ask_question_type() -> str:
    """
    Küsib kasutajalt küsimuste tüübi.

    Returns
    -------
    str
        Küsimuse tüüp: multiple või boolean.
    """
    while True:
        question_type = input("Sisesta küsimuste tüüp (multiple/boolean): ").strip().lower()

        if question_type in QUESTION_TYPES:
            return question_type

        print("Valida saab ainult 'multiple' või 'boolean'.")

In [147]:
categories = load_categories()
print_categories(categories)

username = ask_username()
category_id, category_name = ask_category(categories)
question_count = ask_question_count()
question_type = ask_question_type()

print()
print("Valitud testi seaded:")
print(f"Kasutaja: {username}")
print(f"Kategooria: {category_name}")
print(f"Küsimuste arv: {question_count}")
print(f"Küsimuste tüüp: {question_type}")

Saadaolevad kategooriad:

9: General Knowledge
10: Entertainment: Books
11: Entertainment: Film
12: Entertainment: Music
13: Entertainment: Musicals & Theatres
14: Entertainment: Television
15: Entertainment: Video Games
16: Entertainment: Board Games
17: Science & Nature
18: Science: Computers
19: Science: Mathematics
20: Mythology
21: Sports
22: Geography
23: History
24: Politics
25: Art
26: Celebrities
27: Animals
28: Vehicles
29: Entertainment: Comics
30: Science: Gadgets
31: Entertainment: Japanese Anime & Manga
32: Entertainment: Cartoon & Animations
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Kasutajanimi ei tohi olla tühi.
Sellist kategooriat ei ole nimekirjas.
Sellist katego

## 1.7 Küsimuste laadimine ja testi käivitamine

Selles osas laen API-st küsimused vastavalt kasutaja valikutele.

Seejärel loon `TriviaGame` objekti ja käivitan testi.

Testi ajal:

- kuvatakse küsimus
- kuvatakse vastusevariandid
- kasutaja valib vastuse numbri
- programm kontrollib, kas vastus oli õige
- lõpus kuvatakse skoor ja õiged vastused

In [148]:
questions = fetch_questions(
    amount=question_count,
    category_id=category_id,
    question_type=question_type
)

game = TriviaGame(
    username=username,
    questions=questions
)

game.run()
game.print_score()
game.print_correct_answers()

Alustame testi. Mängija: 1

Küsimus 1: Pete Townshend was the first musician to destroy their intrument live on stage.
1. False
2. True
Vale. Õige vastus oli: False

Teie skoor: 0/1

Õiged vastused:
1. Pete Townshend was the first musician to destroy their intrument live on stage.
   Õige vastus: False


## 1.8 Tulemuse salvestamine CSV-faili

Selles osas loon funktsiooni, mis salvestab testi tulemuse CSV-faili.

CSV-faili salvestatakse:

- kasutajanimi
- kategooria
- küsimuste arv
- küsimuste tüüp
- skoor
- kuupäev

Kui faili veel ei ole, siis programm loob selle ja lisab ka päiserea.

In [149]:
def save_result(result: TriviaResult) -> None:
    """
    Salvestab ühe testi tulemuse CSV-faili.

    Parameters
    ----------
    result : TriviaResult
        Salvestatav testi tulemus.
    """
    file_exists = RESULTS_FILE.exists()

    with RESULTS_FILE.open(mode="a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=CSV_FIELDNAMES)

        if not file_exists:
            writer.writeheader()

        writer.writerow(result.to_csv_row())

In [150]:
result = game.create_result()
save_result(result)

print("Tulemus salvestati CSV-faili.")
print(result)

Tulemus salvestati CSV-faili.
1 | Entertainment: Music | 0/1 | 0.0%


## 1.9 Tulemuste statistika

Selles osas loen CSV-failist kõik varasemad tulemused tagasi sisse.

Seejärel kuvan:

- TOP 5 parimat tulemust
- keskmise tulemuse protsentides

TOP 5 sorteerin protsendi järgi, mitte ainult skoori järgi, sest küsimuste arv võib olla erinev.
Näiteks `4/5` on parem tulemus kui `5/10`.

In [151]:
def read_results() -> list[dict]:
    """
    Loeb tulemused CSV-failist.

    Returns
    -------
    list[dict]
        CSV-failist loetud tulemuste nimekiri.
    """
    if not RESULTS_FILE.exists():
        return []

    with open(RESULTS_FILE, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        results = list(reader)

    return results


def add_percentage_to_results(results: list[dict]) -> list[dict]:
    """
    Lisab igale tulemusele protsendi.

    Parameters
    ----------
    results : list[dict]
        CSV-failist loetud tulemused.

    Returns
    -------
    list[dict]
        Tulemused koos protsendiga.
    """
    results_with_percentage = []

    for result in results:
        try:
            question_count = int(result["küsimuste_arv"])
            score = int(result["skoor"])

            if question_count <= 0:
                continue

            result_with_percentage = result.copy()
            result_with_percentage["küsimuste_arv"] = question_count
            result_with_percentage["skoor"] = score
            result_with_percentage["protsent"] = round(
                score / question_count * 100,
                2,
            )

            results_with_percentage.append(result_with_percentage)

        except (KeyError, ValueError):
            continue

    return results_with_percentage

In [152]:
def get_top_results(results: list[dict], count: int = DEFAULT_TOP_COUNT) -> list[dict]:
    """
    Leiab parimad tulemused protsendi järgi.

    Parameters
    ----------
    results : list[dict]
        Tulemuste nimekiri.
    count : int
        Mitu tulemust tagastatakse.

    Returns
    -------
    list[dict]
        Parimad tulemused.
    """
    return sorted(
        results,
        key=lambda result: (result["protsent"], result["skoor"]),
        reverse=True
    )[:count]


def get_average_percentage(results: list[dict]) -> float:
    """
    Arvutab kõikide tulemuste keskmise protsendi.

    Parameters
    ----------
    results : list[dict]
        Tulemuste nimekiri.

    Returns
    -------
    float
        Keskmine tulemus protsendina.
    """
    if not results:
        return 0.0

    total = sum(result["protsent"] for result in results)
    return round(total / len(results), 2)


def print_top_results(results: list[dict]) -> None:
    """
    Väljastab TOP tulemused ekraanile.

    Parameters
    ----------
    results : list[dict]
        Tulemuste nimekiri.
    """
    if not results:
        print("Tulemusi pole veel salvestatud.")
        return

    print("TOP 5 tulemused:")
    print()

    for index, result in enumerate(results, start=1):
        print(
            f'{index}. {result["kasutajanimi"]} | '
            f'{result["kategooria"]} | '
            f'{result["skoor"]}/{result["küsimuste_arv"]} | '
            f'{result["protsent"]}% | '
            f'{result["kuupäev"]}'
        )

In [153]:
results = read_results()
results = add_percentage_to_results(results)

top_results = get_top_results(results)
average_percentage = get_average_percentage(results)

print_top_results(top_results)

print()
print(f"Keskmine tulemus: {average_percentage}%")

TOP 5 tulemused:

1. c | Entertainment: Music | 1/1 | 100.0% | 2026-05-01 19:04:05
2. d | Entertainment: Musicals & Theatres | 1/1 | 100.0% | 2026-05-01 19:04:27
3. e | Geography | 1/1 | 100.0% | 2026-05-01 19:04:44
4. t | Entertainment: Music | 1/1 | 100.0% | 2026-05-01 19:05:39
5. y | Geography | 1/1 | 100.0% | 2026-05-01 19:12:54

Keskmine tulemus: 55.56%


# Ülesanne 2. Nutika kodu andurite jälgimissüsteem (IoT)

Selles ülesandes loon nutika kodu andurite jälgimissüsteemi.

Programm kasutab:

- CSV-faili andurite toorandmete hoidmiseks
- YAML-faili süsteemi konfiguratsiooni ja piirangute hoidmiseks
- JSON-faili häirete logi salvestamiseks
- Pandast andmete analüüsimiseks

Süsteem loeb anduriandmed, kontrollib väärtuseid YAML-is määratud piiride põhjal, salvestab häired JSON-faili ning koostab Pandase abil päringud ja aruanded.

## Konstandid

In [ ]:
SENSOR_DATA_FILE = Path("sensor_data.csv")
IOT_CONFIG_FILE = Path("iot_config.yaml")
ALARMS_FILE = Path("alarms.json")

DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"

LAST_DAYS_FOR_LIGHT = 7
LAST_HOURS_FOR_MOTION = 48
LAST_DAYS_FOR_HUMIDITY = 14
RECENT_ALARM_COUNT = 5

## 2.1 Failide loomine

Kõigepealt loon kolm faili:

1. `sensor_data.csv` — andurite toorandmed
2. `iot_config.yaml` — andurite konfiguratsioon ja piirangud
3. `alarms.json` — häirete logi

CSV-failis on andurite mõõtmised.  
YAML-failis on kirjas andurite nimed, asukohad, lubatud piirid ja süsteemi seaded.  
JSON-faili salvestatakse häired, kui väärtus ületab YAML-is määratud piiri.

In [ ]:
sensor_rows = [
    ["TEMP_01", "temperature", 21.5, "C", "living room", "2026-03-01 10:00:00"],
    ["TEMP_02", "temperature", 31.5, "C", "kitchen", "2026-03-02 14:23:00"],
    ["TEMP_03", "temperature", 15.2, "C", "bedroom", "2026-03-03 08:15:00"],
    ["TEMP_04", "temperature", 26.4, "C", "garage", "2026-03-04 19:30:00"],

    ["HUM_01", "humidity", 42.0, "%", "living room", "2026-03-05 12:00:00"],
    ["HUM_02", "humidity", 68.5, "%", "bathroom", "2026-03-08 09:45:00"],
    ["HUM_03", "humidity", 76.2, "%", "bedroom", "2026-03-10 21:10:00"],
    ["HUM_04", "humidity", 38.4, "%", "kitchen", "2026-03-12 16:40:00"],

    ["LIGHT_01", "light", 350, "lux", "living room", "2026-03-06 11:15:00"],
    ["LIGHT_02", "light", 120, "lux", "bedroom", "2026-03-09 20:00:00"],
    ["LIGHT_03", "light", 780, "lux", "kitchen", "2026-03-11 13:30:00"],
    ["LIGHT_04", "light", 950, "lux", "garage", "2026-03-13 18:25:00"],

    ["MOT_01", "motion", 1, "1/0", "living room", "2026-03-13 22:10:00"],
    ["MOT_02", "motion", 0, "1/0", "bedroom", "2026-03-14 07:30:00"],
    ["MOT_03", "motion", 1, "1/0", "garage", "2026-03-14 23:50:00"],
    ["MOT_04", "motion", 1, "1/0", "kitchen", "2026-03-15 01:20:00"],
]